# Day 1.5 — Build the Agent Loop Manually

One hardcoded tool interaction cannot handle a question that needs two tools, or none, or
an unknown number. The mechanism that makes an application *agentic* is a loop:

```text
model -> decide -> tool -> observation -> model -> ... -> final answer
```

We are going to write that loop by hand, one line at a time, watch the message list grow,
and make it hit its own step limit. Only at the very end do we open the packaged version
in `src/` and confirm it is the same twenty-odd lines.

## Before you begin

### Learning outcomes

- Write the model/tool/observation loop yourself, with a step counter and a stop condition.
- Read the message list after every turn and say why each message is there.
- Prove the limit belongs to your code by making the loop stop at `max_steps`.

Architecture reference: [D04](../../diagrams/source/day_01.md).

### Expected observation

Each turn appends one `assistant` message and one `tool` message per executed tool. With
`max_steps=1` the run ends unfinished and says so, instead of looping forever.

## Concept briefing

## Why the application owns termination

After one tool result, the model may ask for another tool or return a final answer. That
creates a loop whose length is not known in advance. It is tempting to write "stop when
finished" in the system message and trust the model. That is not an execution limit. A
confused model can repeat the same request, alternate between tools, or continue refining
an already adequate answer. Each turn consumes time, tokens and money.

Host code therefore enforces a maximum number of steps. Reaching the limit is not the
same as crashing. A good runtime returns a visible status such as `max_steps` together
with the partial trace. Reporting incomplete work honestly is safer than pretending the
run completed.

## Error compounding

Multi-step systems amplify small error rates. Suppose, only for illustration, that each
model decision has a 95% chance of being acceptable and that errors are independent. The
chance that ten decisions are all acceptable is:

```text
0.95 ^ 10 = approximately 0.60
```

The independence assumption is simplistic, but the lesson is useful: a system with many
model decisions can be much less reliable than any single impressive response suggests.
This motivates bounded loops, deterministic validation, fewer calls, clear tools and
evaluation of complete trajectories rather than isolated answers.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Pick a provider and look at the tools

`MockModelProvider` follows the same interface as the real one: give it messages and tool
definitions, get back a `ModelTurn` with either text or tool requests. That is why the
identical loop works in both modes.

In [ ]:
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

# Never construct the live provider unless a key exists - it raises without one.
provider = OpenRouterProvider() if LIVE else MockModelProvider()
print("Provider:", type(provider).__name__)

tools = default_tool_registry()
tool_definitions = [tool.definition for tool in tools.values()]
print()
print("Tools this agent may request:")
for name, tool in tools.items():
    print(f"  {name:20} {tool.definition.description}")

### Step 2 — Seed the conversation

Two messages start every run: the standing instructions, and the question. Everything else
in the list will be produced by the loop.

In [ ]:
from research_agent.agent import SYSTEM_MESSAGE

question = "Explain an AI agent using the local notes and calculate 12 * 7."

messages = [
    Message(role="system", content=SYSTEM_MESSAGE),
    Message(role="user", content=question),
]

print("SYSTEM_MESSAGE (what the model is told once, up front):")
print(SYSTEM_MESSAGE)
print("Messages before the loop starts:", len(messages))

### Step 3 — Do one turn by hand

Call the model. It answers with **either** text (it is finished) **or** tool requests (it
wants something done first). Deciding which of those happened is the branch the whole loop
is built around.

In [ ]:
turn = provider.complete(messages, tool_definitions)

print("turn.content    :", repr(turn.content))
print("turn.tool_calls :", [(call.name, call.arguments) for call in turn.tool_calls])
print()
print("Finished?", "yes - this is the final answer" if not turn.tool_calls else "no - it wants a tool first")

# Whatever it said, it becomes part of the conversation.
messages.append(Message(role="assistant", content=turn.content, tool_calls=turn.tool_calls))
print("Messages now:", len(messages))

### Step 4 — Execute the request and append the observation

`tool.execute` validates the arguments and returns a `ToolResult` instead of raising, so a
bad request becomes an observation the model can read and recover from. Note the
`tool_call_id`: it is what ties the answer to the question.

In [ ]:
for call in turn.tool_calls:
    tool = tools.get(call.name)
    if tool is None:
        output = f"Tool error: unknown tool '{call.name}'"
    else:
        output = tool.execute(call.id, call.arguments).output
    print(f"executed {call.name}{call.arguments} -> {output[:80]}")
    messages.append(
        Message(role="tool", name=call.name, tool_call_id=call.id, content=output)
    )

print()
print("Messages now:", len(messages))
for index, message in enumerate(messages):
    print(f"  {index}. {message.role}")

### Step 5 — Now write the whole loop

Steps 3 and 4 repeat until the model stops asking for tools. Three things make it an
agent loop rather than a `while True`:

1. a **step counter** with a hard maximum;
2. a **branch** on `turn.tool_calls` — text means finished;
3. **validation** of the final answer before we call the run a success.

Read the loop, then read the printed trace under it.

In [ ]:
from pydantic import ValidationError
from research_agent.schemas import ResearchResponse

def run_agent(question, max_steps=5, show_messages=True):
    """The whole agent, in about 25 lines."""
    messages = [Message(role="system", content=SYSTEM_MESSAGE),
                Message(role="user", content=question)]

    for step in range(1, max_steps + 1):                      # 1. bounded, never while True
        turn = provider.complete(messages, tool_definitions)
        messages.append(Message(role="assistant", content=turn.content,
                                tool_calls=turn.tool_calls))

        if not turn.tool_calls:                               # 2. no tools = final answer
            try:
                answer = ResearchResponse.model_validate_json(turn.content)
            except ValidationError as error:                  # 3. validate before trusting
                return {"status": "failed", "steps": step, "messages": messages,
                        "error": f"Final response failed validation: {error}"}
            return {"status": "completed", "steps": step, "messages": messages,
                    "answer": answer}

        for call in turn.tool_calls:                          # execute what was requested
            tool = tools.get(call.name)
            output = (tool.execute(call.id, call.arguments).output if tool
                      else f"Tool error: unknown tool '{call.name}'")
            messages.append(Message(role="tool", name=call.name,
                                    tool_call_id=call.id, content=output))

        if show_messages:
            print(f"--- after model turn {step}: {len(messages)} messages ---")
            for index, message in enumerate(messages):
                requested = [c.name for c in message.tool_calls]
                print(f"   {index}. role={message.role:9} tool_requests={requested}")

    return {"status": "max_steps", "steps": max_steps, "messages": messages,
            "error": f"Agent stopped after {max_steps} steps"}

result = run_agent(question)
print()
print("status:", result["status"], "| model turns:", result["steps"])

### Step 6 — Read the result

The loop ended because the model returned text instead of a tool request, and that text
passed the schema. Both facts are checked by our code, not asserted by the model.

In [ ]:
answer = result.get("answer")
if answer is not None:
    print(answer.model_dump_json(indent=2))
    print()
    print("Tools the run actually used:", answer.tools_used)
else:
    print("No validated answer. error:", result["error"])

print()
print("Observations the model saw:")
for message in result["messages"]:
    if message.role == "tool":
        print(f"  {message.name:20} -> {message.content[:70]}")

### Step 7 — Force the limit

Give the same two-tool question a budget of one turn. The model cannot possibly finish,
and the honest outcome is `max_steps` plus a partial trace — not a crash, and not a
pretend answer.

In [ ]:
limited = run_agent(question, max_steps=1, show_messages=False)

print("status :", limited["status"])
print("steps  :", limited["steps"])
print("error  :", limited.get("error"))
print("messages produced:", len(limited["messages"]))
print()
print("Compare with the unlimited run:", result["status"], "in", result["steps"], "turns.")
print("The step limit lives in OUR for-loop. No instruction to the model can enforce it.")

### Step 8 — The same loop, packaged

`AgentRunner` in `src/research_agent/agent.py` is the loop you just wrote, plus two extras:
it records token usage, and it refuses to run the *same* tool with the *same* arguments
twice (a stuck model would otherwise burn the whole budget). Open the file and match it
line by line against Step 5.

In [ ]:
from research_agent.agent import AgentRunner

runner = AgentRunner(provider=provider, tools=tools, max_steps=5)
packaged = runner.run(question)

print("Our loop      :", result["status"], "in", result["steps"], "turns,",
      len(result["messages"]), "messages")
print("AgentRunner   :", packaged.status, "in", packaged.steps, "turns,",
      len(packaged.messages), "messages")
print("Usage recorded:", packaged.usage.model_dump())
print()
print("Same trace, same stopping rule - the only new thing is the bookkeeping.")

### Try it yourself

What happens if the loop forgets to append the `tool` observation? Predict it, then run the
worked solution.

In [ ]:
# --- Worked solution ---
# Prediction: the model asks for the tool, we run it, and then we throw the answer away.
# On the next turn the conversation looks exactly as it did before, so the model asks for
# the same thing again... and again, until the step limit stops it. The observation is not
# a log line; it is the only way a result gets back into the model's context.

def run_agent_forgetting_observations(question, max_steps=4):
    messages = [Message(role="system", content=SYSTEM_MESSAGE),
                Message(role="user", content=question)]
    for step in range(1, max_steps + 1):
        turn = provider.complete(messages, tool_definitions)
        messages.append(Message(role="assistant", content=turn.content,
                                tool_calls=turn.tool_calls))
        if not turn.tool_calls:
            return {"status": "completed", "steps": step}
        for call in turn.tool_calls:
            tool = tools.get(call.name)
            output = tool.execute(call.id, call.arguments).output if tool else "?"
            print(f"turn {step}: model asked for {call.name}{call.arguments} -> {output[:40]}")
            # (the Message(role="tool", ...) append is deliberately missing)
    return {"status": "max_steps", "steps": max_steps}

broken = run_agent_forgetting_observations(question)
print()
print("Result:", broken)
print("Every turn repeats the same request, and only max_steps ends it.")
print("This is exactly the failure AgentRunner's duplicate-request check catches early.")

### Checkpoint

**1. Where is the stopping rule, and why can it not live in the system prompt?**

<details><summary>Show answer</summary>

It is the `for step in range(1, max_steps + 1)` header in your own Python. A system prompt
is a request to a text generator; a confused or adversarial model can ignore it, and every
extra turn costs time and tokens. Only host code can *guarantee* the loop ends — which is
also why hitting the limit returns the status `max_steps` and the partial trace instead of
raising.

</details>

**2. A two-tool question produced how many messages, and what is each of them for?**

<details><summary>Show answer</summary>

Seven — count them in the Step 8 output. `system` (standing instructions) and `user` (the
question) start the list. Then each of the two tool turns adds an `assistant` message
holding the request and a `tool` message holding the observation, which is four more.
Finally one `assistant` message carries the JSON answer. Nothing is optional: drop the
`tool` messages and the model never learns the results, as the worked solution above
demonstrates.

</details>

### Recap

- **Limitation we saw:** a single request/execute/respond sequence cannot handle an unknown
  number of steps, and nothing in it can stop a model that keeps asking.
- **Layer we added:** a bounded loop we wrote ourselves — step counter, branch on
  `tool_calls`, observation messages, and validation of the final answer.
- **Evidence it worked:** the trace printed after every turn, the two-tool question
  completed and validated, and `max_steps=1` ended the run honestly with a partial trace.